# 15. Tableau 최종 산출 테이블 생성

13번에서 확정한 `classification_detail_final`과 14번에서 생성한 `topic_group`을 기준으로, 원본 `intellytics_display_online_voc` row에 `memo_id`, `topic_group`, `pred_topic`만 추가한 Tableau 연동용 테이블을 생성합니다.

- 중간 산출: `classification_tableau_final`
- 최종 Tableau 연결 권장: `classification_tableau_grouped_final`


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import ml.final_classification_builder as final_classification_builder
import taxonomy.topic_group_generator as topic_group_generator

importlib.reload(config_loader)
importlib.reload(final_classification_builder)
importlib.reload(topic_group_generator)

from common.config_loader import load_config, get_output_table, get_reference_table, get_source_table
from ml.final_classification_builder import build_tableau_final_df, save_tableau_final
from taxonomy.topic_group_generator import build_and_save_tableau_grouped_final

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")

print("source =", get_source_table(config, "raw_review_table"))
print("final_detail =", get_output_table(config, "classification_detail_final"))
print("topic_group =", get_output_table(config, "topic_group"))
print("tableau_final =", get_output_table(config, "classification_tableau_final"))
print("tableau_grouped_final =", get_output_table(config, "classification_tableau_grouped_final"))


In [ ]:
# 사전 검증: detail/topic_group row 수 확인
final_detail_table = get_output_table(config, "classification_detail_final")
topic_group_table = get_output_table(config, "topic_group")

check = {
    "final_detail_rows": spark.table(final_detail_table).where(F.col("prompt_version") == config["version"]["prompt_version"]).count(),
    "topic_group_rows": spark.table(topic_group_table).where(F.col("prompt_version") == config["version"]["prompt_version"]).count(),
}

if check["final_detail_rows"] == 0:
    raise ValueError("classification_detail_final is empty. Run notebook 13 first.")
if check["topic_group_rows"] == 0:
    raise ValueError("topic_group is empty. Run notebook 14 first.")

check


In [ ]:
# 원본 row + memo_id + 최종 topic 테이블 생성
tableau_df = build_tableau_final_df(spark, config)
tableau_rows = tableau_df.count()
tableau_distinct_memo_ids = tableau_df.select("memo_id").dropDuplicates().count()

tableau_table = save_tableau_final(
    spark,
    config,
    tableau_df,
    write_mode="replace_version",
)

{
    "tableau_table": tableau_table,
    "tableau_rows": tableau_rows,
    "tableau_distinct_memo_ids": tableau_distinct_memo_ids,
}


In [ ]:
# topic_group까지 붙인 Tableau 최종 테이블 생성
grouped_result = build_and_save_tableau_grouped_final(
    spark,
    config,
    write_mode="replace_version",
)

grouped_result


In [ ]:
# Tableau 최종 테이블 분포 확인
grouped_table = get_output_table(config, "classification_tableau_grouped_final")
category_mapping_table = get_reference_table(config, "category_mapping_table")

grouped_df = spark.table(grouped_table)

display(
    grouped_df.alias("t")
    .join(
        spark.table(category_mapping_table).alias("m"),
        on=["cate_1_depth", "cate_2_depth"],
        how="left",
    )
    .groupBy(
        "t.cate_1_depth",
        "m.cate_1_depth_kor",
        "t.cate_2_depth",
        "m.cate_2_depth_kor",
        "t.sc_measurement",
        "t.topic_group",
        "t.pred_topic",
    )
    .agg(
        F.count("*").alias("raw_row_cnt"),
        F.countDistinct("memo_id").alias("distinct_memo_id_cnt"),
    )
    .orderBy("t.cate_1_depth", "t.cate_2_depth", "t.sc_measurement", "t.topic_group", F.desc("raw_row_cnt"))
)


In [ ]:
# Tableau 연결 전 샘플 확인
display(
    grouped_df.select(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "year",
        "country",
        "brand_name",
        "device_type",
        "memo_id",
        "memo",
        "topic_group",
        "pred_topic",
    )
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", "topic_group")
    .limit(100)
)
